In [1]:
from pathlib import Path
import sys, json, hashlib, importlib.util, zipfile
ROOT = Path("/home/featurize/work/biological_prior_validation").resolve()
if not ROOT.is_dir():
    raise FileNotFoundError(f"Please change ROOT to the existing project directory：{ROOT}")
for required in [ROOT / "src/evaluation.py", ROOT / "src/development_split.py"]:
    if not required.is_file():
        raise FileNotFoundError(f"Please complete Step 3/4 first: missing {required}")


In [2]:
SOURCE = '"""Four-context exploratory reproduction; frozen development selection, final B scoring."""\nfrom pathlib import Path\nimport json,hashlib,sys,importlib.util,os,time\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom scipy.sparse.linalg import LinearOperator,cg\nfrom scipy.special import xlogy\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.cluster import MiniBatchKMeans\nfrom sklearn.neighbors import NearestNeighbors\ntry:import faiss\nexcept ImportError:faiss=None\nCONTEXTS=[(\'A172\',\'Lapatinib\'),(\'A172\',\'Trametinib\'),(\'T98G\',\'Nintedanib\'),(\'U87MG\',\'ZSTK474\')]\nREPS=[\'replicate_1\',\'replicate_2\'];METHODS=[\'CellOnly\',\'FixedPPI\',\'AdaptivePPI\']\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n\ndef digest(p):\n h=hashlib.sha256()\n with Path(p).open(\'rb\') as f:\n  for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n return h.hexdigest()\ndef js(p):return json.loads(Path(p).read_text())\ndef save(p,o):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_name(p.name+\'.tmp\');t.write_text(json.dumps(o,indent=2,ensure_ascii=False,allow_nan=False,default=str));t.replace(p)\ndef freeze(p,o):\n o=json.loads(json.dumps(o,allow_nan=False,default=str))\n if Path(p).exists() and js(p)!=o:raise ValueError(f\'Frozen configuration changed: {p}\')\n save(p,o)\ndef import_file(name,p):\n s=importlib.util.spec_from_file_location(name,p);m=importlib.util.module_from_spec(s);sys.modules[name]=m;s.loader.exec_module(m);return m\n\ndef defaults(root):\n root=Path(root)\n return {\'project_root\':str(root),\'run_path\':str(root/\'data/splits/step2_q05_seed20261008_v1\'),\n \'development_name\':\'inner_q05_seed20261008_v1\',\'experiment_name\':\'step5_four_contexts_v1\',\'seed\':20261008,\n \'alpha_grid\':[0.,.0001,.0003,.001,.003,.01,.03,.1,.3,1.,3.,10.,30.,100.,300.],\n \'fixed_beta_grid\':sorted(set([0.]+np.logspace(-6,-3,7).tolist()+np.logspace(-3,0,13).tolist()+np.logspace(0,2,5).tolist())),\n \'adaptive_beta_grid\':[0.,.001,.003162277660168379,.01,.03162277660168379,.1,.3162277660168379],\n \'tau_quantiles\':[.25,.5,.75,.9,.97],\'tau_high_multiplier\':3.,\'svd_components\':50,\'knn_k\':30,\n \'target_cells_per_metacell\':1050,\'min_metacells\':12,\'max_metacells\':80,\'min_members\':30,\n \'cg_rtol\':1e-6,\'cg_maxiter\':500,\'max_outer\':60,\'outer_x_rtol\':1e-5,\'outer_s_atol\':1e-4,\n \'batch_cells\':256,\'run_final_evaluation\':True}\n\ndef check_config(c):\n for k in [\'alpha_grid\',\'fixed_beta_grid\',\'adaptive_beta_grid\']:\n  v=np.asarray(c[k],float)\n  if v.ndim!=1 or not len(v) or not np.isfinite(v).all() or (v<0).any() or 0 not in v or len(np.unique(v))!=len(v):raise ValueError(f\'Invalid {k}, including required zero\')\n for k in [\'svd_components\',\'knn_k\',\'target_cells_per_metacell\',\'min_metacells\',\'max_metacells\',\'min_members\',\'cg_maxiter\',\'max_outer\',\'batch_cells\']:\n  if not isinstance(c[k],int) or c[k]<1:raise ValueError(f\'Invalid {k}\')\n if c[\'min_metacells\']>c[\'max_metacells\']:raise ValueError(\'Invalid metacell limits\')\n if not isinstance(c[\'seed\'],int) or c[\'seed\']<0:raise ValueError(\'Invalid seed\')\n if any(not 0<q<1 for q in c[\'tau_quantiles\']):raise ValueError(\'Invalid quantiles\')\n for k in [\'cg_rtol\',\'outer_x_rtol\',\'outer_s_atol\',\'tau_high_multiplier\']:\n  if not np.isfinite(c[k]) or c[k]<=0:raise ValueError(f\'Invalid {k}\')\n if Path(c[\'experiment_name\']).name!=c[\'experiment_name\'] or c[\'experiment_name\'] in [\'.\',\'..\']:raise ValueError(\'Invalid experiment name\')\n\ndef ppi_operator(genes,edges):\n mapping={x:i for i,x in enumerate(genes.gene_symbol)}\n e=edges.loc[edges.gene_a.isin(mapping)&edges.gene_b.isin(mapping)].copy()\n if e.duplicated([\'gene_a\',\'gene_b\']).any() or e.gene_a.eq(e.gene_b).any():raise ValueError(\'PPI duplicate/self edges\')\n u=e.gene_a.map(mapping).to_numpy(dtype=int);v=e.gene_b.map(mapping).to_numpy(dtype=int);w=e.ppi_weight.to_numpy(dtype=float)\n if not len(w) or not np.isfinite(w).all() or (w<=0).any():raise ValueError(\'Missing/invalid induced PPI edges\')\n g=len(genes);W=sp.csr_matrix((np.r_[w,w],(np.r_[u,v],np.r_[v,u])),shape=(g,g));L=sp.diags(np.asarray(W.sum(1)).ravel())-W\n vals,V=np.linalg.eigh(L.toarray());scale=float(vals.max())\n if vals.min()<-1e-7 or scale<=0:raise ValueError(\'Invalid PPI Laplacian\')\n return {\'u\':u,\'v\':v,\'w\':w/scale,\'gamma\':np.maximum(vals,0)/scale,\'V\':V,\'scale\':scale,\'n_genes\':g}\n\ndef laplacian(ppi,weights):\n u,v=ppi[\'u\'],ppi[\'v\'];g=ppi[\'n_genes\'];W=sp.csr_matrix((np.r_[weights,weights],(np.r_[u,v],np.r_[v,u])),shape=(g,g))\n return (sp.diags(np.asarray(W.sum(1)).ravel())-W).tocsr()\n\ndef representation(Y,c,seed):\n n,g=Y.shape\n if n<max(3,c[\'min_metacells\']) or g<2:raise ValueError(\'Insufficient cells/genes\')\n svd=TruncatedSVD(n_components=min(c[\'svd_components\'],n-1,g-1),n_iter=5,n_oversamples=10,random_state=seed)\n Z=svd.fit_transform(Y).astype(np.float32);norm=np.linalg.norm(Z,axis=1,keepdims=True);Z=np.divide(Z,norm,out=np.zeros_like(Z),where=norm>0)\n requested=int(np.clip(round(n/c[\'target_cells_per_metacell\']),c[\'min_metacells\'],c[\'max_metacells\']))\n for factor in [1.,.85,.70,.55]:\n  k=min(n,max(c[\'min_metacells\'],round(requested*factor)))\n  km=MiniBatchKMeans(n_clusters=k,batch_size=min(8192,max(1024,n//5)),random_state=seed,n_init=3,reassignment_ratio=.01)\n  labels=km.fit_predict(Z);used,labels=np.unique(labels,return_inverse=True);counts=np.bincount(labels)\n  if len(counts)>=2 and counts.min()>=c[\'min_members\']:break\n else:raise ValueError(\'No metacell assignment satisfies pre-fixed minimum size\')\n k=min(c[\'knn_k\'],n-1);search=min(n,k+8)\n if faiss is not None:\n  faiss.omp_set_num_threads(1);idx=faiss.IndexHNSWFlat(Z.shape[1],32,faiss.METRIC_INNER_PRODUCT);idx.hnsw.efConstruction=80;idx.hnsw.efSearch=64;idx.add(np.ascontiguousarray(Z));similar,neighbors=idx.search(np.ascontiguousarray(Z),search);backend=\'FAISS HNSW, one thread\'\n else:\n  if n>10000:raise RuntimeError(\'Install faiss-cpu in the active kernel; refusing quadratic brute-force on a large context\')\n  nn=NearestNeighbors(n_neighbors=search,metric=\'cosine\',algorithm=\'brute\',n_jobs=1).fit(Z);dist,neighbors=nn.kneighbors(Z);similar=1-dist;backend=\'sklearn exact cosine, small data only\'\n nonself=neighbors!=np.arange(n)[:,None];select=nonself & (np.cumsum(nonself,axis=1)<=k)\n if not (select.sum(axis=1)==k).all():raise ValueError(\'kNN neighbor count mismatch\')\n W=sp.csr_matrix((np.clip(similar[select],0,1),(np.repeat(np.arange(n),k),neighbors[select])),shape=(n,n));W=W.maximum(W.T);W.setdiag(0);W.eliminate_zeros()\n P=sp.csr_matrix((np.ones(n),(np.arange(n),labels)),shape=(n,len(counts)))\n wm=(P.T@W@P).toarray();wm=(wm+wm.T)/2;np.fill_diagonal(wm,0);lc=np.diag(wm.sum(1))-wm\n inv=1/np.sqrt(counts);lc=inv[:,None]*lc*inv[None,:];vals,U=np.linalg.eigh(lc)\n if vals.min()<-1e-7 or vals.max()<=0:raise ValueError(\'Degenerate/non-PSD Galerkin cell operator\')\n vals=np.maximum(vals,0)/vals.max();mean=(P.T@Y).toarray()/counts[:,None]\n C=U.T@(np.sqrt(counts)[:,None]*mean)\n return {\'labels\':labels,\'counts\':counts,\'U\':U,\'lc\':vals,\'C\':C,\'backend\':backend,\'svd_explained_variance\':float(np.nan_to_num(svd.explained_variance_ratio_).sum()),\'knn_edges\':int(W.nnz//2)}\n\ndef fit_fixed(r,ppi,alpha,beta):\n if beta==0:return r[\'C\']/(1+alpha*r[\'lc\'][:,None])\n q=r[\'C\']@ppi[\'V\'];den=1+alpha*r[\'lc\'][:,None]+beta*ppi[\'gamma\'][None,:]\n return (q/den)@ppi[\'V\'].T\n\ndef edge_energy(X,ppi):\n out=np.empty(len(ppi[\'u\']))\n for lo in range(0,len(out),2048):\n  diff=X[:,ppi[\'u\'][lo:lo+2048]]-X[:,ppi[\'v\'][lo:lo+2048]];out[lo:lo+2048]=np.einsum(\'ij,ij->j\',diff,diff)\n return out\n\ndef solve_gene(C,lc,alpha,L,beta,c,X0=None):\n if beta==0:return C/(1+alpha*lc[:,None])\n shape=C.shape;a=1+alpha*lc;diag=a[:,None]+beta*L.diagonal()[None,:]\n def mv(z):\n  x=z.reshape(shape);return (a[:,None]*x+beta*(L@x.T).T).ravel()\n op=LinearOperator((C.size,C.size),matvec=mv,dtype=float);pre=LinearOperator(op.shape,matvec=lambda z:z/diag.ravel(),dtype=float)\n x,info=cg(op,C.ravel(),x0=None if X0 is None else X0.ravel(),M=pre,rtol=c[\'cg_rtol\'],atol=0,maxiter=c[\'cg_maxiter\'])\n residual=np.linalg.norm(mv(x)-C.ravel())/max(np.linalg.norm(C),1e-12)\n if info!=0 or residual>max(5*c[\'cg_rtol\'],1e-8):raise RuntimeError(f\'CG failed: info={info}, residual={residual}\')\n return x.reshape(shape)\n\ndef tau_value(spec,r,ppi,alpha,c):\n if spec==\'infinity\':return np.inf\n positive=edge_energy(fit_fixed(r,ppi,alpha,0),ppi);positive=positive[positive>1e-12]\n if not len(positive):return np.inf\n if spec==\'high\':return float(c[\'tau_high_multiplier\']*np.quantile(positive,.99))\n return float(np.quantile(positive,float(spec)))\n\ndef fit_adaptive(r,ppi,alpha,beta,tau,c):\n X=fit_fixed(r,ppi,alpha,beta);s=np.ones(len(ppi[\'u\']))\n if beta==0 or not np.isfinite(tau):return X,s,True,0\n if tau<=0:raise ValueError(\'Invalid adaptive tau\')\n s=np.clip(1-edge_energy(X,ppi)/tau,0,1)\n for it in range(1,c[\'max_outer\']+1):\n  new=solve_gene(r[\'C\'],r[\'lc\'],alpha,laplacian(ppi,ppi[\'w\']*s),beta,c,X)\n  sn=np.clip(1-edge_energy(new,ppi)/tau,0,1)\n  dx=np.linalg.norm(new-X)/max(np.linalg.norm(X),1e-12);ds=float(np.max(np.abs(sn-s)));X=new;s=sn\n  if dx<=c[\'outer_x_rtol\'] and ds<=c[\'outer_s_atol\']:return X,s,True,it\n return X,s,False,c[\'max_outer\']\n\ndef metacell_prediction(r,X):return (r[\'U\']@X)/np.sqrt(r[\'counts\'])[:,None]\n\nclass DevelopmentLoss:\n """Exact per-cell Poisson loss by sufficient statistics, with floor-boundary fallback."""\n def __init__(self,V,library,labels,q,scale,floor):\n  self.V=V.tocsr().astype(float);self.lib=np.asarray(library,float);self.labels=np.asarray(labels,int);self.q=q;self.scale=scale;self.floor=floor\n  if (self.lib<=0).any() or not 0<q<1 or floor<=0:raise ValueError(\'Invalid development exposure\')\n  m=int(labels.max())+1;n,g=V.shape;self.n_pairs=n*g\n  P=sp.csr_matrix((np.ones(n),(labels,np.arange(n))),shape=(m,n))\n  self.bs=(P@self.V).toarray();blog=self.V.copy();blog.data=xlogy(blog.data,blog.data)\n  self.blog=(P@blog).toarray();self.blib=(P@(sp.diags(np.log(self.lib))@self.V)).toarray()\n  self.exposure=np.bincount(labels,weights=self.lib,minlength=m);self.n=np.bincount(labels,minlength=m)\n  self.minimum=np.array([self.lib[labels==j].min() for j in range(m)]);self.maximum=np.array([self.lib[labels==j].max() for j in range(m)])\n def __call__(self,pred):\n  pred=np.asarray(pred,float)\n  if pred.shape!=self.bs.shape or not np.isfinite(pred).all():raise ValueError(\'Invalid development prediction\')\n  with np.errstate(over=\'raise\'):rate=np.expm1(np.maximum(pred,0))/self.scale*(1-self.q)/self.q\n  high=rate*self.minimum[:,None]>=self.floor;low=rate*self.maximum[:,None]<=self.floor;mix=~(high|low)\n  result=np.zeros_like(rate)\n  result[high]=(2*(self.blog-self.blib-self.bs*xlogy(np.ones_like(rate),np.maximum(rate,np.finfo(float).tiny))-self.bs+rate*self.exposure[:,None]))[high]\n  result[low]=(2*(self.blog-self.bs*np.log(self.floor)-self.bs+self.n[:,None]*self.floor))[low]\n  for group in np.unique(np.where(mix)[0]):\n   cols=np.where(mix[group])[0];rows=np.where(self.labels==group)[0];b=self.V[rows][:,cols].toarray();mu=np.maximum(self.lib[rows,None]*rate[group,cols][None,:],self.floor)\n   result[group,cols]=2*np.sum(xlogy(b,b/mu)-b+mu,axis=0)\n  value=float(result.sum()/self.n_pairs)\n  if not np.isfinite(value) or value<-1e-7:raise ValueError(\'Invalid development loss\')\n  return max(value,0)\n\ndef tune(r,ppi,loss,c):\n rows=[]\n def record(method,alpha,beta,ts,X,converged=True,iterations=0):\n  value=loss(metacell_prediction(r,X)) if converged else None\n  row={\'method\':method,\'alpha\':float(alpha),\'beta\':float(beta),\'tau_rule\':str(ts),\'development_deviance\':value,\'converged\':bool(converged),\'iterations\':int(iterations)};rows.append(row);return row\n cell=[]\n for a in c[\'alpha_grid\']:cell.append(record(\'CellOnly\',a,0,\'infinity\',fit_fixed(r,ppi,a,0)))\n bestcell=min(cell,key=lambda x:(x[\'development_deviance\'],x[\'alpha\']));alpha=bestcell[\'alpha\']\n fixed=[]\n for b in c[\'fixed_beta_grid\']:fixed.append(record(\'FixedPPI\',alpha,b,\'infinity\',fit_fixed(r,ppi,alpha,b)))\n bestfixed=min(fixed,key=lambda x:(x[\'development_deviance\'],x[\'beta\']))\n adaptive=[]\n for b in sorted(set(c[\'adaptive_beta_grid\']+[bestfixed[\'beta\']])):\n  for rule in ([\'infinity\'] if b==0 else [str(q) for q in c[\'tau_quantiles\']]+[\'high\',\'infinity\']):\n   tau=tau_value(rule,r,ppi,alpha,c);X,s,ok,it=fit_adaptive(r,ppi,alpha,b,tau,c)\n   adaptive.append(record(\'AdaptivePPI\',alpha,b,rule,X,ok,it))\n valid=[x for x in adaptive if x[\'converged\']]\n if not valid:raise RuntimeError(\'No converged adaptive candidate\')\n besta=min(valid,key=lambda x:(x[\'development_deviance\'],x[\'beta\'],x[\'tau_rule\']!=\'infinity\',x[\'tau_rule\']))\n return {m:row for m,row in zip(METHODS,[bestcell,bestfixed,besta])},pd.DataFrame(rows)\n\ndef context_id(line,drug,rep):return f\'{line}__{drug}__{rep}\'\n\ndef verified_sparse(folder,name):\n rec=js(Path(folder)/\'receipt.json\');expected=next(x[\'sha256\'] for x in rec[\'files\'] if x[\'name\']==name);p=Path(folder)/name\n if digest(p)!=expected:raise ValueError(f\'Source/cache SHA mismatch: {p}\')\n return sp.load_npz(p).tocsr(),expected\n\ndef context_counts(run,dev,roles,genes,line,drug,rep,phase,target=False):\n size=int(js(run/\'identity.json\')[\'cells_per_block\']);cols=genes.gene_col.to_numpy(dtype=int);pieces=[];ids=[];libs=[];sources=[]\n eligible=roles.inner_evaluation_eligible if phase==\'inner\' else roles.evaluation_eligible\n mask=(eligible & roles.cell_line.eq(line) & roles.drug.eq(drug) & roles.replicate.eq(rep)).to_numpy()\n for b in range((len(roles)+size-1)//size):\n  lo=b*size;hi=min(lo+size,len(roles));keep=mask[lo:hi]\n  if not keep.any():continue\n  folder=(dev if phase==\'inner\' else run)/f\'block_{b:05d}\'\n  name=(\'A_validation.npz\' if target else \'A_train.npz\') if phase==\'inner\' else \'A_input.npz\'\n  if target and phase!=\'inner\':raise ValueError(\'Final B must only be loaded by evaluate_final\')\n  a,sha=verified_sparse(folder,name)\n  library=np.asarray(a.sum(axis=1)).ravel().astype(np.int64)\n  if not target and (library[keep]<=0).any():raise ValueError(\'Positive input library required\')\n  pieces.append(a[keep][:,cols]);ids.extend(np.arange(lo,hi)[keep]);libs.extend(library[keep]);sources.append({\'path\':str(folder/name),\'sha256\':sha})\n if not pieces:raise ValueError(f\'Empty requested context: {line}/{drug}/{rep}\')\n return sp.vstack(pieces,format=\'csr\'),np.asarray(ids,dtype=np.int64),np.asarray(libs,dtype=np.int64),sources\n\ndef norm_counts(a,library,scale):\n y=(sp.diags(scale/library)@a.astype(float)).tocsr();y.data=np.log1p(y.data);return y\n\ndef build_models(c):\n check_config(c);root=Path(c[\'project_root\']).resolve();run=Path(c[\'run_path\']).resolve();dev=run/\'development\'/c[\'development_name\']\n if not run.is_relative_to(root/\'data/splits\'):raise ValueError(\'Run must be inside project data/splits\')\n pair=js(run/\'A_preprocessing/evaluation_pair_contract.json\');metric=js(run/\'evaluation/metric_contract.json\');ds=js(dev/\'step4_summary.json\');dc=js(dev/\'scenario_A_contract.json\')\n if ds[\'status\']!=\'step4_prepared\' or ds[\'final_B_accessed\'] or ds[\'models_fitted\']:raise ValueError(\'Unexpected Step 4 state\')\n if digest(run/\'evaluation/metric_contract.json\')!=ds[\'metric_contract_sha256_after\']:raise ValueError(\'Metric changed since Step 4\')\n if metric[\'pair_contract_sha256\']!=digest(run/\'A_preprocessing/evaluation_pair_contract.json\'):raise ValueError(\'Pair contract changed\')\n cp=run/\'A_preprocessing\'/pair[\'cells_file\'];gp=run/\'A_preprocessing\'/pair[\'genes_file\']\n if digest(cp)!=pair[\'cells_sha256\'] or digest(gp)!=pair[\'genes_sha256\']:raise ValueError(\'Frozen outer IDs changed\')\n roles=pd.read_parquet(dev/\'cell_roles.parquet\').sort_values(\'matrix_row\').reset_index(drop=True);cells=pd.read_parquet(cp).sort_values(\'matrix_row\').reset_index(drop=True)\n if not np.array_equal(roles.matrix_row,np.arange(len(roles))) or len(cells)!=len(roles) or not cells.cell_barcode.eq(roles.cell_barcode).all() or not cells.evaluation_eligible.eq(roles.evaluation_eligible).all():raise ValueError(\'Step 4 cell roles mismatch\')\n out=run/\'step5\'/c[\'experiment_name\'];out.mkdir(parents=True,exist_ok=True)\n edgepath=root/\'data/processed/ppi_annotation_reference_edges.parquet\'\n init={\'config\':{k:v for k,v in c.items() if k!=\'run_final_evaluation\'},\'source_sha256\':ENGINE_SHA,\'metric_sha256\':digest(run/\'evaluation/metric_contract.json\'),\n \'pair_sha256\':digest(run/\'A_preprocessing/evaluation_pair_contract.json\'),\'step4_identity_sha256\':digest(dev/\'identity.json\'),\'cell_roles_sha256\':digest(dev/\'cell_roles.parquet\'),\n \'ppi_sha256\':digest(edgepath),\'contexts\':CONTEXTS,\'scope\':\'four preselected exploratory contexts, both replicate labels; NOT cross-replicate evaluation\',\n \'tuning\':\'old sequential alpha(CellOnly), then fixed beta; adaptive alpha fixed to CellOnly; beta grid + fixed optimum, tau training-derived rule\',\n \'environment\':{\'numpy\':np.__version__,\'pandas\':pd.__version__,\'scipy\':__import__(\'scipy\').__version__,\'sklearn\':__import__(\'sklearn\').__version__,\'faiss\':None if faiss is None else getattr(faiss,\'__version__\',\'present\')}}\n # JSON round-trip normalizes tuples to lists for immutable comparison.\n init=json.loads(json.dumps(init));freeze(out/\'experiment_contract.json\',init)\n cfgout=root/\'configs\'/f"{c[\'experiment_name\']}.json";freeze(cfgout,init)\n edges=pd.read_parquet(edgepath);outergenes=pd.read_parquet(gp).sort_values(\'selected_col\')\n registries=[];ppi_cache={}\n for rep in REPS:\n  innerpath=dev/f\'{rep}_inner_selected_genes.parquet\';expected=next(x[\'selected_genes_sha256\'] for x in ds[\'inner_genes_by_replicate\'] if x[\'replicate\']==rep)\n  if digest(innerpath)!=expected:raise ValueError(\'Inner gene universe changed\')\n  innergenes=pd.read_parquet(innerpath).sort_values(\'selected_col\')\n  for line,drug in CONTEXTS:\n   key=context_id(line,drug,rep);folder=out/key;folder.mkdir(exist_ok=True);receipt=folder/\'model_receipt.json\'\n   if receipt.exists():\n    rec=js(receipt)\n    for file in rec[\'files\']+rec[\'input_sources\']:\n     path=Path(file[\'path\'])\n     if digest(path)!=file[\'sha256\']:raise ValueError(f\'Frozen model/source changed: {path}\')\n    registries.append(rec);print(\'Reuse frozen models:\',key,flush=True);continue\n   if (run/\'evaluation/target_access_lock.json\').exists():raise ValueError(\'Final B already accessed; no new fitting allowed in this run\')\n   seed=c[\'seed\']+1000*CONTEXTS.index((line,drug))+10*REPS.index(rep)\n   cachekey=digest(innerpath)\n   if cachekey not in ppi_cache:ppi_cache[cachekey]=ppi_operator(innergenes,edges)\n   ppi=ppi_cache[cachekey]\n   T,ids,lt,sources=context_counts(run,dev,roles,innergenes,line,drug,rep,\'inner\')\n   r=representation(norm_counts(T,lt,metric[\'normalization_scale\']),c,seed)\n   # Grids and training graphs are frozen/built before opening development targets.\n   freeze(folder/\'development_input_scope.json\',{\'matrix_row\':ids.tolist(),\'inner_gene_sha256\':cachekey,\'input_sources\':sources,\'representation_source\':\'A_train only\'})\n   V,vids,lv,vsources=context_counts(run,dev,roles,innergenes,line,drug,rep,\'inner\',target=True)\n   if not np.array_equal(ids,vids):raise ValueError(\'Development common-pair row mismatch\')\n   loss=DevelopmentLoss(V,lt,r[\'labels\'],dc[\'inner_q\'],metric[\'normalization_scale\'],metric[\'mu_floor\'])\n   chosen,curves=tune(r,ppi,loss,c);curves.to_csv(folder/\'development_grid.csv\',index=False)\n   save(folder/\'chosen_parameters.json\',chosen)\n   np.savez_compressed(folder/\'inner_representation.npz\',matrix_row=ids,labels=r[\'labels\'],counts=r[\'counts\'],lc=r[\'lc\'],U=r[\'U\'],gene_idx=innergenes.gene_idx.to_numpy())\n   print(\'Development selected:\',key,chosen,flush=True)\n   del T,V,r,loss\n   cachekey=digest(gp)\n   if cachekey not in ppi_cache:ppi_cache[cachekey]=ppi_operator(outergenes,edges)\n   ppif=ppi_cache[cachekey]\n   A,rows,la,asources=context_counts(run,dev,roles,outergenes,line,drug,rep,\'outer\')\n   rf=representation(norm_counts(A,la,metric[\'normalization_scale\']),c,seed)\n   graphinfo={\'context\':key,\'n_cells\':len(rows),\'n_outer_genes\':len(outergenes),\'n_inner_genes\':len(innergenes),\'n_metacells\':len(rf[\'counts\']),\'metacell_min\':int(rf[\'counts\'].min()),\'metacell_max\':int(rf[\'counts\'].max()),\'knn_backend\':rf[\'backend\'],\'knn_edges\':rf[\'knn_edges\'],\'PPI_edges\':len(ppif[\'w\']),\'PPI_scale\':ppif[\'scale\'],\'svd_explained_variance\':rf[\'svd_explained_variance\'],\'all_input\':\'full A only\',\'inner_gene_hash\':digest(innerpath),\'outer_gene_hash\':digest(gp)}\n   np.savez_compressed(folder/\'outer_representation.npz\',matrix_row=rows,labels=rf[\'labels\'],counts=rf[\'counts\'],lc=rf[\'lc\'],U=rf[\'U\'],gene_idx=outergenes.gene_idx.to_numpy())\n   for method in METHODS:\n    par=chosen[method];weights=np.ones(len(ppif[\'w\']));ok=True;it=0\n    if method==\'AdaptivePPI\':\n     tau=tau_value(par[\'tau_rule\'],rf,ppif,par[\'alpha\'],c);X,weights,ok,it=fit_adaptive(rf,ppif,par[\'alpha\'],par[\'beta\'],tau,c)\n    else:tau=np.inf;X=fit_fixed(rf,ppif,par[\'alpha\'],par[\'beta\'])\n    if not ok:raise RuntimeError(f\'Final A-only adaptive fit did not converge: {key}; final B remains unopened\')\n    pred=metacell_prediction(rf,X)\n    if not np.isfinite(pred).all():raise ValueError(\'Nonfinite final prediction\')\n    np.savez_compressed(folder/f\'{method}.npz\',matrix_row=rows,labels=rf[\'labels\'],prediction=pred,gene_idx=outergenes.gene_idx.to_numpy(),library_A=la,edge_s=weights)\n    graphinfo[method+\'_iterations\']=it;graphinfo[method+\'_effective_edges\']=int(np.count_nonzero(ppif[\'w\']*weights));graphinfo[method+\'_tau\']=None if not np.isfinite(tau) else tau\n   save(folder/\'graph_and_scope.json\',graphinfo)\n   files=[{\'path\':str(p.resolve()),\'sha256\':digest(p)} for p in sorted(folder.iterdir()) if p.is_file() and p.name!=\'model_receipt.json\']\n   rec={\'context_id\':key,\'cell_line\':line,\'drug\':drug,\'replicate\':rep,\'n_cells\':len(rows),\'n_genes\':len(outergenes),\'n_pairs\':len(rows)*len(outergenes),\'files\':files,\'input_sources\':sources+vsources+asources}\n   save(receipt,rec);registries.append(rec);print(\'Frozen final A-only models:\',key,flush=True)\n   del A,rf\n scope=pd.DataFrame([{k:x[k] for k in [\'context_id\',\'cell_line\',\'drug\',\'replicate\',\'n_cells\',\'n_genes\',\'n_pairs\']} for x in registries]);scope.to_csv(out/\'scope_coverage.csv\',index=False)\n freeze(out/\'frozen_model_registry.json\',{\'experiment_contract_sha256\':digest(out/\'experiment_contract.json\'),\'outer_gene_sha256\':digest(gp),\'models\':registries,\'scored_contexts\':CONTEXTS,\'all_contexts_evaluated\':False,\'methods\':METHODS})\n roster=cells.loc[cells.evaluation_eligible].groupby([\'cell_line\',\'drug\',\'replicate\'],observed=True).size().rename(\'n_cells\').reset_index();roster[\'execution_status\']=[\'planned_four_contexts\' if (a,b) in CONTEXTS else \'not_run_in_Step5\' for a,b in zip(roster.cell_line,roster.drug)];roster.to_csv(out/\'all_context_status.csv\',index=False)\n devmod=import_file(\'step5_environment_helper\',root/\'src/development_split.py\');devmod.save_environment(root,out)\n return out\n\ndef evaluate_final(c,out):\n run=Path(c[\'run_path\']);root=Path(c[\'project_root\']);out=Path(out);registry=js(out/\'frozen_model_registry.json\');metric=js(run/\'evaluation/metric_contract.json\');pair=js(run/\'A_preprocessing/evaluation_pair_contract.json\')\n if registry[\'experiment_contract_sha256\']!=digest(out/\'experiment_contract.json\'):raise ValueError(\'Frozen registry contract changed\')\n ev=import_file(\'step5_fixed_evaluator\',root/\'src/evaluation.py\')\n if ev.ENGINE_SHA!=metric[\'engine_sha256\']:raise ValueError(\'Step 3 evaluation source changed\')\n for rec in registry[\'models\']:\n  for f in rec[\'files\']+rec[\'input_sources\']:\n   if digest(f[\'path\'])!=f[\'sha256\']:raise ValueError(\'Frozen files changed before/after target access\')\n gp=run/\'A_preprocessing\'/pair[\'genes_file\']\n if digest(gp)!=registry[\'outer_gene_sha256\'] or digest(gp)!=pair[\'genes_sha256\']:raise ValueError(\'Final gene IDs changed\')\n genes=pd.read_parquet(gp).sort_values(\'selected_col\');cp=run/\'A_preprocessing\'/pair[\'cells_file\']\n if digest(cp)!=pair[\'cells_sha256\']:raise ValueError(\'Final cell IDs changed\')\n cells=pd.read_parquet(cp).sort_values(\'matrix_row\').reset_index(drop=True);size=js(run/\'identity.json\')[\'cells_per_block\']\n lock={\'kind\':\'step5_exploratory_four_contexts\',\'experiment_path\':str(out.resolve()),\'registry_sha256\':digest(out/\'frozen_model_registry.json\'),\'metric_sha256\':digest(run/\'evaluation/metric_contract.json\'),\n       \'n_scope_cells\':sum(x[\'n_cells\'] for x in registry[\'models\']),\'n_scope_pairs\':sum(x[\'n_pairs\'] for x in registry[\'models\']),\'all_contexts_scored\':False,\'later_changes\':\'no retuning or replacement of this frozen experiment; later studies must disclose reuse of exploratory B targets\'}\n # Write lock BEFORE the first B read. Same frozen experiment can retry computation.\n freeze(run/\'evaluation/target_access_lock.json\',lock);freeze(out/\'final_target_access_lock.json\',lock)\n loaded={}\n for rec in registry[\'models\']:\n  folder=out/rec[\'context_id\']\n  loaded[rec[\'context_id\']]={}\n  for method in METHODS:\n   with np.load(folder/f\'{method}.npz\',allow_pickle=False) as z:loaded[rec[\'context_id\']][method]={k:z[k] for k in z.files}\n pieces=[];selected=genes.gene_col.to_numpy(dtype=int)\n for b in range((len(cells)+size-1)//size):\n  lo=b*size;hi=min(lo+size,len(cells));meta=cells.iloc[lo:hi]\n  mask=meta.evaluation_eligible.to_numpy() & np.array([(a,d) in CONTEXTS for a,d in zip(meta.cell_line,meta.drug)])\n  if not mask.any():continue\n  folder=run/f\'block_{b:05d}\';B,_=verified_sparse(folder,\'B_target.npz\');A,_=verified_sparse(folder,\'A_input.npz\');lb=np.asarray(B.sum(1)).ravel();la=np.asarray(A.sum(1)).ravel()\n  if B.shape!=A.shape or len(B.indptr)-1!=len(meta):raise ValueError(\'Final count block dimensions mismatch\')\n  for rec in registry[\'models\']:\n   take=mask & meta.cell_line.eq(rec[\'cell_line\']).to_numpy() & meta.drug.eq(rec[\'drug\']).to_numpy() & meta.replicate.eq(rec[\'replicate\']).to_numpy();local=np.flatnonzero(take)\n   for start in range(0,len(local),c[\'batch_cells\']):\n    rows=local[start:start+c[\'batch_cells\']];globalrows=rows+lo\n    for method in METHODS:\n     model=loaded[rec[\'context_id\']][method];where=np.searchsorted(model[\'matrix_row\'],globalrows)\n     if not np.array_equal(model[\'matrix_row\'][where],globalrows) or not np.array_equal(model[\'gene_idx\'],genes.gene_idx.to_numpy()) or not np.array_equal(model[\'library_A\'][where],la[rows]):raise ValueError(\'Prediction cell/gene/exposure mismatch\')\n     pred=model[\'prediction\'][model[\'labels\'][where]]\n     scores=ev.score_rows(B[rows][:,selected],pred,la[rows],lb[rows],metric[\'q\'],metric[\'normalization_scale\'],metric[\'mu_floor\'])\n     sums=scores.sum(numeric_only=True).to_dict();sums.update(method=method,cell_line=rec[\'cell_line\'],drug=rec[\'drug\'],replicate=rec[\'replicate\'],n_cells=len(rows));pieces.append(sums)\n  print(f\'Final B score block {b+1}\',flush=True)\n frame=pd.DataFrame(pieces);nums=[\'deviance_sum\',\'normalized_sse_sum\',\'n_pairs\',\'n_cells\',\'mu_floor_pairs\',\'negative_log_predictions\'];table=frame.groupby([\'cell_line\',\'drug\',\'replicate\',\'method\'],as_index=False)[nums].sum()\n for x in registry[\'models\']:\n  part=table.loc[table.cell_line.eq(x[\'cell_line\']) & table.drug.eq(x[\'drug\']) & table.replicate.eq(x[\'replicate\'])]\n  if len(part)!=len(METHODS) or not part.n_pairs.eq(x[\'n_pairs\']).all() or not part.n_cells.eq(x[\'n_cells\']).all():raise ValueError(\'Final common-I coverage incomplete\')\n table[\'mean_poisson_deviance\']=table.deviance_sum/table.n_pairs;table[\'normalized_mse\']=table.normalized_sse_sum/table.n_pairs\n base=table.loc[table.method.eq(\'CellOnly\'),[\'cell_line\',\'drug\',\'replicate\',\'mean_poisson_deviance\']].rename(columns={\'mean_poisson_deviance\':\'CellOnly_deviance\'})\n table=table.merge(base,on=[\'cell_line\',\'drug\',\'replicate\'],validate=\'many_to_one\');table[\'delta_deviance\']=table.mean_poisson_deviance-table.CellOnly_deviance\n table[\'relative_deviance_percent\'],table[\'relative_status\']=ev.relative_change(table.mean_poisson_deviance,table.CellOnly_deviance,metric[\'relative_min_denominator\'])\n params=[]\n for rec in registry[\'models\']:\n  chosen=js(out/rec[\'context_id\']/\'chosen_parameters.json\')\n  for method,par in chosen.items():params.append({\'cell_line\':rec[\'cell_line\'],\'drug\':rec[\'drug\'],\'replicate\':rec[\'replicate\'],\'method\':method,**par})\n table=table.merge(pd.DataFrame(params),on=[\'cell_line\',\'drug\',\'replicate\',\'method\'],validate=\'one_to_one\')\n table.to_csv(out/\'baseline_reproduction.csv\',index=False);table.to_csv(root/\'results/raw\'/f"{c[\'experiment_name\']}_baseline_reproduction.csv",index=False)\n return table\n\ndef comparison_report(c,out,table=None):\n root=Path(c[\'project_root\']);out=Path(out)\n old={(\'U87MG\',\'ZSTK474\'):[-.0672,-.1484,-.1033,-.0453],(\'A172\',\'Lapatinib\'):[-.0259,-.0524,-.0362,-.0602],(\'A172\',\'Trametinib\'):[0.,-.1581,.0184,.0132],(\'T98G\',\'Nintedanib\'):[.0122,-.1140,-.0660,-.1075]}\n historical=[]\n for (line,drug),values in old.items():\n  for direction,f,a in [(\'replicate_1_to_2\',values[0],values[1]),(\'replicate_2_to_1\',values[2],values[3])]:\n   for method,value in [(\'FixedPPI\',f),(\'AdaptivePPI\',a)]:historical.append({\'cell_line\':line,\'drug\':drug,\'direction\':direction,\'method\':method,\'old_relative_normalized_mse_percent\':value,\'source\':\'uploaded manuscript Table 1, page 16; rounded reported values\',\'protocol\':\'cross-label expression holdout; biological independence unresolved in current audit\',\'absolute_error_available\':False,\'directly_comparable_to_new_primary\':False})\n pd.DataFrame(historical).to_csv(out/\'historical_manuscript_reference.csv\',index=False)\n scope=pd.read_csv(out/\'scope_coverage.csv\');graphs=[js(out/x/\'graph_and_scope.json\') for x in scope.context_id]\n differences=[\n  [\'data population\',\'initial global analysis: 1,052,205 cells; detailed cross-label context sizes require old raw output\',f\'current outer eligible population retained from Step2; selected four-context scope={int(scope.n_cells.sum())}\', \'author metadata/QC and context restrictions differ\'],\n  [\'target and metric\',\'held-out normalized expression squared error\',\'held-out B mean Poisson deviance primary; normalized B MSE secondary\',\'units/target differ; do not subtract historical % from new primary %\'],\n  [\'replicate protocol\',\'historical detailed table: 1→2 and 2→1\',\'within each replicate label, train/validate by molecules, final B\',\'Scenario B blocked; label analyses are not historical directions\'],\n  [\'gene selection\',\'old selected genes from globally normalized expression\',\'inner train-only per-label genes; final frozen full-A genes\',\'gene universe/input scope differs; exact gene overlap requires old gene IDs\'],\n  [\'cell graph\',\'old strict code: train-only SVD50, cosine HNSW k30; initial global code shares representation before split\',\'same SVD/kNN/Galerkin rule, constructed separately from T or full A\', \'final B never constructs graph; training scopes differ\'],\n  [\'metacells\',\'old strict code target1050 cells, range12–80; initial global analysis 1000\',f"new context fits actual counts={\',\'.join(str(x[\'n_metacells\']) for x in graphs)}",\'context-specific sizes and fold re-fitting differ\'],\n  [\'normalization\',\'log1p(10000*C/library_C) on old processed matrix\',\'T/full-A all-gene library, logCP10K; inverse predicts validation/B exposure\',\'development and final exposure mappings kept distinct\'],\n  [\'hyperparameters\',\'CellOnly alpha, then fixed beta; adaptive alpha held at CellOnly optimum\',\'same sequential order; tune on exact inner per-cell Poisson deviance\',\'old criterion MSE vs new deviance; Task2 still needs joint fixed/adaptive tuning\'],\n  [\'adaptive threshold\',\'training edge energy quantiles plus high and infinity\',\'same weighting formula, training-derived quantile rule refitted on final A\',\'rule transfers across inner/final feature refits; numeric tau can differ\'],\n  [\'PPI\',\'initial manuscript graph: 7,460 undirected edges\',\'induced current PPI edge counts recorded per context\',\'raw annotation mapping and A-based genes differ\'],\n  [\'old run availability\',\'manuscript rounded percentages and supplied source code\',\'new raw scores/parameters/scopes saved\',\'exact old runtime outputs were not supplied; cannot claim exact numeric reproduction\']]\n diff=pd.DataFrame(differences,columns=[\'item\',\'old_pipeline\',\'new_pipeline\',\'reason_or_limit\']);diff.to_csv(out/\'pipeline_difference_table.csv\',index=False)\n (root/\'results/summary\').mkdir(parents=True,exist_ok=True);diff.to_csv(root/\'results/summary\'/f"{c[\'experiment_name\']}_pipeline_differences.csv",index=False)\n report=\'\'\'# Step 5 reproduction review\\n\\nFour contexts are exploratory selections from the original manuscript. All methods and labels are retained.\\n\\nThe numerical solver and adaptive weighting structure follow the supplied original implementation. The old manuscript cross-label MSE table is saved as a historical reference, not a same-protocol rerun. Exact old raw result files/selected gene identifiers are needed to audit exact old numerical reproduction.\\n\\nThe new protocol is within-replicate molecular holdout. It cannot verify cross-biological-replicate generalization. Old/new errors and percentages use different targets and units, so their direct difference cannot establish a method improvement or a discrepancy threshold.\\n\\nThe fixed/adaptive models here preserve the original sequential-alpha tuning for reproduction. This is not the fully jointly optimized Fixed PPI comparator required for Task 2. Do not draw biological PPI-specific conclusions before matched attenuation and degree-preserving randomization controls.\\n\\nReview pipeline_difference_table.csv, development_grid.csv files, graph_and_scope.json files and baseline_reproduction.csv before proceeding. If unexpected differences occur, examine sample scope, gene overlap, normalization, leakage, parameter grids and metacells first. Do not change target-driven contexts or tune final B.\\n\'\'\'\n if table is not None:report+=\'\\nFinal B has been accessed under a frozen experiment registry. Computational retries require identical predictions/configuration. Later controls reuse these exploratory targets and must disclose that reuse; they do not become new confirmatory evidence.\\n\'\n else:report+=\'\\nDevelopment/model preparation is complete; final B was not scored in this execution.\\n\'\n (root/\'reports\').mkdir(exist_ok=True);(root/\'reports\'/f"{c[\'experiment_name\']}_review.md").write_text(report)\n summary={\'status\':\'four_context_baseline_scored\' if table is not None else \'models_frozen_final_B_not_scored\',\'experiment_path\':str(out),\'n_scope_cells\':int(scope.n_cells.sum()),\'n_scope_pairs\':int(scope.n_pairs.sum()),\'contexts\':CONTEXTS,\'replicate_labels\':REPS,\'cross_replicate_evaluation\':\'blocked_pending_provenance\',\'old_exact_numerical_reproduction\':\'not_verified_without_original_raw_outputs\',\'old_new_direct_error_comparison\':\'not_valid_different_protocols\',\'final_B_accessed\':table is not None,\'requires_review_before_Task2\':True,\'new_all_contexts_claim\':False}\n save(out/\'step5_summary.json\',summary);save(root/\'results/raw\'/f"{c[\'experiment_name\']}_summary.json",summary)\n return summary\n\ndef run(c):\n out=build_models(c)\n table=evaluate_final(c,out) if c[\'run_final_evaluation\'] else None\n summary=comparison_report(c,out,table)\n return out,table,summary\n'
PLOT_SOURCE = '"""Figures are always regenerated from saved CSV data."""\nfrom pathlib import Path\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\ndef plot_results(csv_file,figure_file):\n table=pd.read_csv(csv_file);fig,axes=plt.subplots(2,1,figsize=(11,8),constrained_layout=True)\n contexts=table[[\'cell_line\',\'drug\',\'replicate\']].drop_duplicates().sort_values([\'cell_line\',\'drug\',\'replicate\'])\n keys=[tuple(x) for x in contexts.to_numpy()];labels=[f\'{a} / {b}\\n{r}\' for a,b,r in keys]\n for method,color,offset in [(\'CellOnly\',\'#264653\',-.22),(\'FixedPPI\',\'#e9a23b\',0),(\'AdaptivePPI\',\'#2a9d8f\',.22)]:\n  sub=table[table.method==method].set_index([\'cell_line\',\'drug\',\'replicate\']).reindex(keys)\n  x=list(range(len(keys)));x=[v+offset for v in x]\n  axes[0].bar(x,sub.mean_poisson_deviance,width=.22,label=method,color=color)\n  axes[1].bar(x,sub.relative_deviance_percent,width=.22,label=method,color=color)\n axes[0].set_ylabel(\'Held-out B mean Poisson deviance\');axes[1].set_ylabel(\'Relative to CellOnly (%)\');axes[1].axhline(0,color=\'#264653\',linewidth=.8)\n for ax in axes:ax.set_xticks(range(len(keys)),labels,rotation=30,ha=\'right\');ax.spines[[\'top\',\'right\']].set_visible(False)\n axes[0].legend(loc=\'lower left\',bbox_to_anchor=(0,1.02),ncol=3,frameon=False);fig.suptitle(\'Four exploratory contexts · within-label molecular holdout\')\n Path(figure_file).parent.mkdir(parents=True,exist_ok=True);fig.savefig(figure_file,dpi=180);plt.close(fig)\n'

def install_source(path,content):
    path.parent.mkdir(parents=True,exist_ok=True)
    data=content.encode("utf-8")
    if path.exists() and path.read_bytes()!=data:
        previous=hashlib.sha256(path.read_bytes()).hexdigest()
        backup=path.with_name(path.name+"."+previous[:12]+".backup")
        if not backup.exists():backup.write_bytes(path.read_bytes())
    path.write_bytes(data)

install_source(ROOT/"src/reproduce.py",SOURCE)
install_source(ROOT/"src/plot_reproduction.py",PLOT_SOURCE)
module_spec=importlib.util.spec_from_file_location("reproduce",ROOT/"src/reproduce.py")
rep=importlib.util.module_from_spec(module_spec)
sys.modules["reproduce"]=rep
module_spec.loader.exec_module(rep)
print("Reproduction engine SHA256:",rep.ENGINE_SHA)


Reproduction engine SHA256: e2952130b31f1b0dc3c473ff40858dfba07007d0973dccbd38a60972a6ab6dff


In [3]:
CONFIG = rep.defaults(ROOT)

CONFIG["run_path"] = str(ROOT/"data/splits/step2_q05_seed20261008")
CONFIG["development_name"] = "inner_q05_seed20261008"
CONFIG["experiment_name"] = "step5_four_contexts"
CONFIG["run_final_evaluation"] = True
rep.check_config(CONFIG)
print(json.dumps(CONFIG,ensure_ascii=False,indent=2))

entry = "from pathlib import Path\nimport json, sys\nROOT=Path(__file__).resolve().parents[1]\nsys.path.insert(0,str(ROOT/'src'))\nimport reproduce\nCONFIG=" + repr(CONFIG) + "\nif __name__=='__main__':\n    out,table,summary=reproduce.run(CONFIG)\n    print(json.dumps(summary,ensure_ascii=False,indent=2))\n"
install_source(ROOT/"experiments/exp01_reproduce_baseline.py",entry)


{
  "project_root": "/home/featurize/work/biological_prior_validation",
  "run_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008",
  "development_name": "inner_q05_seed20261008",
  "experiment_name": "step5_four_contexts",
  "seed": 20261008,
  "alpha_grid": [
    0.0,
    0.0001,
    0.0003,
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0,
    30.0,
    100.0,
    300.0
  ],
  "fixed_beta_grid": [
    0.0,
    1e-06,
    3.162277660168379e-06,
    9.999999999999999e-06,
    3.1622776601683795e-05,
    0.0001,
    0.00031622776601683794,
    0.001,
    0.0017782794100389228,
    0.0031622776601683794,
    0.005623413251903491,
    0.01,
    0.01778279410038923,
    0.03162277660168379,
    0.056234132519034905,
    0.1,
    0.1778279410038923,
    0.31622776601683794,
    0.5623413251903491,
    1.0,
    3.1622776601683795,
    10.0,
    31.622776601683793,
    100.0
  ],
  "adaptive_beta_grid": [
    0.0,

In [4]:
TEST_SOURCE = "from pathlib import Path\nimport sys,tempfile,json,shutil\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom scipy.special import kl_div\nfrom unittest.mock import patch\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/'src'))\nimport reproduce as r\n\ndef reject(fn):\n try:fn()\n except (ValueError,RuntimeError):return\n raise AssertionError('Expected rejection')\n\ndef math_tests():\n checks=[];rng=np.random.default_rng(12)\n genes=pd.DataFrame({'gene_symbol':['a','b','c','d']})\n edges=pd.DataFrame({'gene_a':['a','b','c'],'gene_b':['b','c','d'],'ppi_weight':[.7,.8,.9]})\n pp=r.ppi_operator(genes,edges);C=rng.normal(size=(3,4));rr={'C':C,'lc':np.array([0.,.3,1.]),'U':np.eye(3),'counts':np.ones(3)}\n cfg=r.defaults('/tmp');cfg.update(max_outer=100,cg_rtol=1e-10)\n L=r.laplacian(pp,pp['w']).toarray();alpha=.3;beta=.2\n oracle=np.linalg.solve(np.kron(np.diag(1+alpha*rr['lc']),np.eye(4))+beta*np.kron(np.eye(3),L),C.ravel()).reshape(3,4)\n fixed=r.fit_fixed(rr,pp,alpha,beta);cg=r.solve_gene(C,rr['lc'],alpha,sp.csr_matrix(L),beta,cfg)\n assert np.allclose(fixed,oracle,rtol=1e-9,atol=1e-9) and np.allclose(cg,oracle,rtol=1e-8,atol=1e-9)\n assert np.array_equal(r.fit_fixed(rr,pp,alpha,0),C/(1+alpha*rr['lc'][:,None]))\n X,s,ok,_=r.fit_adaptive(rr,pp,alpha,beta,np.inf,cfg);assert ok and np.allclose(X,fixed) and np.all(s==1)\n X,s,ok,_=r.fit_adaptive(rr,pp,alpha,0,.1,cfg);assert ok and np.array_equal(X,r.fit_fixed(rr,pp,alpha,0))\n X,s,ok,_=r.fit_adaptive(rr,pp,alpha,beta,.5,cfg);assert ok and ((s>=0)&(s<=1)).all()\n checks.append('dense Kronecker oracle, spectral/CG equivalence, beta-zero and infinity nesting, support-preserving adaptive weights')\n # Exact sufficient-statistic score vs cell-level oracle, including mixed floors.\n V=sp.csr_matrix([[0,2,0],[3,0,1],[0,0,0],[1,2,4]]);lib=np.array([1,10,3,8]);labels=np.array([0,0,1,1]);pred=np.array([[1e-9,1.2,-.5],[0.,.7,2.]])\n for q in [.2,.5,.8]:\n  for floor in [1e-8,.1,1.]:\n   loss=r.DevelopmentLoss(V,lib,labels,q,10.,floor)\n   mu=np.maximum(np.expm1(np.maximum(pred[labels],0))*lib[:,None]/10*(1-q)/q,floor)\n   oracle=float((2*kl_div(V.toarray(),mu)).mean());actual=loss(pred)\n   assert np.isclose(actual,oracle,rtol=1e-10,atol=1e-10),(actual,oracle)\n checks.append('fast development loss equals dense per-cell Poisson oracle, zero targets, negative predictions, mixed floors, q scaling')\n reject(lambda:r.DevelopmentLoss(V,np.array([0,10,3,8]),labels,.5,10,1e-8))\n bad={**cfg,'fixed_beta_grid':[.1]};reject(lambda:r.check_config(bad))\n checks.append('invalid exposure and missing beta-zero grids rejected')\n return checks\n\ndef integration_fixture(root,source_base):\n # Reuse shipped Step 4 synthetic fixture, never user data.\n spec=__import__('importlib.util',fromlist=['']).spec_from_file_location('step4_fixture_tests',source_base.parent/'step4_development/tests/test_development_split.py')\n t=__import__('importlib.util',fromlist=['']).module_from_spec(spec);spec.loader.exec_module(t)\n cfg,A=t.fixture(root);run=Path(cfg['run_path']);(root/'src').mkdir();(root/'results/raw').mkdir(parents=True)\n shutil.copy(source_base.parent/'step4_development/src/development_split.py',root/'src/development_split.py');shutil.copy(source_base.parent/'step3_evaluation/src/evaluation.py',root/'src/evaluation.py')\n metric=t.d.read_json(run/'evaluation/metric_contract.json');metric.update(engine_sha256=t.d.digest(root/'src/evaluation.py'),relative_min_denominator=1e-12)\n t.d.save_json(run/'evaluation/metric_contract.json',metric)\n for b in range(3):\n  folder=run/f'block_{b:05d}';B=sp.csr_matrix(np.random.default_rng(b+41).poisson(7,size=(40,8)));sp.save_npz(folder/'B_target.npz',B)\n  rec=t.d.read_json(folder/'receipt.json');rec['files'].append({'name':'B_target.npz','sha256':t.d.digest(folder/'B_target.npz')});t.d.save_json(folder/'receipt.json',rec)\n t.d.prepare_development(cfg)\n allgenes=pd.read_parquet(run/'genes_all.parquet');edge=[]\n for i in range(8):\n  for j in range(i+1,8):edge.append({'gene_a':allgenes.gene_symbol[i],'gene_b':allgenes.gene_symbol[j],'ppi_weight':.8})\n (root/'data/processed').mkdir();pd.DataFrame(edge).to_parquet(root/'data/processed/ppi_annotation_reference_edges.parquet',index=False)\n c=r.defaults(root);c.update(run_path=str(run),alpha_grid=[0.,.3],fixed_beta_grid=[0.,.01],adaptive_beta_grid=[0.,.01],tau_quantiles=[.5],svd_components=2,knn_k=3,min_metacells=2,max_metacells=2,min_members=1,max_outer=60,batch_cells=3)\n return c\n\ndef tests(source_base):\n checks=math_tests()\n with tempfile.TemporaryDirectory() as temp:\n  root=Path(temp);cfg=integration_fixture(root,source_base);run=Path(cfg['run_path']);original=sp.load_npz\n  def guard(path,*a,**k):\n   if Path(path).name=='B_target.npz':assert (run/'evaluation/target_access_lock.json').exists() and (run/'step5'/cfg['experiment_name']/'frozen_model_registry.json').exists()\n   if Path(path).name=='C_counts.npz':raise AssertionError('C should never be loaded')\n   return original(path,*a,**k)\n  with patch.object(sp,'load_npz',guard):out,table,summary=r.run(cfg)\n  assert summary['final_B_accessed'] and len(table)==len(r.CONTEXTS)*len(r.REPS)*len(r.METHODS)\n  assert table.n_cells.eq(4).all() and table.n_pairs.eq(16).all()\n  assert table.loc[table.method.eq('CellOnly'),'relative_deviance_percent'].eq(0).all()\n  checks.append('full four-context x two-label x three-method integration, fixed registry before B, C never loaded, common-I counts')\n  # Recompute final scores by an independent direct loop from fixture B and stored models.\n  for rec in r.js(out/'frozen_model_registry.json')['models']:\n   for method in r.METHODS:\n    with np.load(out/rec['context_id']/f'{method}.npz') as z:\n     rows=z['matrix_row'];prediction=z['prediction'][z['labels']];library=z['library_A']\n    genes=pd.read_parquet(run/'A_preprocessing/selected_genes_A.parquet').gene_col.to_numpy()\n    B=sp.vstack([sp.load_npz(run/f'block_{b:05d}/B_target.npz') for b in range(3)]).toarray()[rows][:,genes]\n    mu=np.maximum(np.expm1(np.maximum(prediction,0))*library[:,None]/10000,1e-8)\n    expected=float((2*kl_div(B,mu)).mean());actual=table.loc[table.cell_line.eq(rec['cell_line']) & table.drug.eq(rec['drug']) & table.replicate.eq(rec['replicate']) & table.method.eq(method),'mean_poisson_deviance'].iloc[0]\n    assert np.isclose(actual,expected,atol=1e-9)\n  checks.append('final scores equal independent cell-level count oracle')\n  with patch.object(sp,'load_npz',guard):out2,table2,s2=r.run(cfg)\n  pd.testing.assert_frame_equal(table,table2)\n  reject(lambda:r.run({**cfg,'alpha_grid':[0.,1.]}))\n  model=out/r.context_id(*r.CONTEXTS[0],r.REPS[0])/'CellOnly.npz';model.write_bytes(b'tampered')\n  reject(lambda:r.run(cfg))\n  checks.append('identical frozen retry, changed grid and altered model rejected after final B access')\n return {'status':'PASS','checks':checks}\n\nif __name__=='__main__':print(json.dumps(tests(Path(__file__).resolve().parents[1]),indent=2))\n"
test_ns={"__name__":"step5_embedded_selftests","__file__":str(ROOT/"tests/test_reproduce.py")}
exec(compile(TEST_SOURCE,"step5_embedded_selftests.py","exec"),test_ns)
TEST_REPORT={"status":"PASS","checks":test_ns["math_tests"]()}
print(json.dumps(TEST_REPORT,ensure_ascii=False,indent=2))


{
  "status": "PASS",
  "checks": [
    "dense Kronecker oracle, spectral/CG equivalence, beta-zero and infinity nesting, support-preserving adaptive weights",
    "fast development loss equals dense per-cell Poisson oracle, zero targets, negative predictions, mixed floors, q scaling",
    "invalid exposure and missing beta-zero grids rejected"
  ]
}


In [5]:
assert TEST_REPORT["status"]=="PASS"
EXPERIMENT_PATH=rep.build_models(CONFIG)
rep.save(EXPERIMENT_PATH/"synthetic_test_report.json",TEST_REPORT)
SCOPE=pd_scope=__import__("pandas").read_csv(EXPERIMENT_PATH/"scope_coverage.csv")
print(SCOPE.to_string(index=False))
REGISTRY=rep.js(EXPERIMENT_PATH/"frozen_model_registry.json")
assert len(REGISTRY["models"])==len(rep.CONTEXTS)*len(rep.REPS)
print("The A-only models of all methods have been frozen:",EXPERIMENT_PATH)


Development selected: A172__Lapatinib__replicate_1 {'CellOnly': {'method': 'CellOnly', 'alpha': 0.003, 'beta': 0.0, 'tau_rule': 'infinity', 'development_deviance': 0.6769137367490631, 'converged': True, 'iterations': 0}, 'FixedPPI': {'method': 'FixedPPI', 'alpha': 0.003, 'beta': 0.0017782794100389228, 'tau_rule': 'infinity', 'development_deviance': 0.6769129384431815, 'converged': True, 'iterations': 0}, 'AdaptivePPI': {'method': 'AdaptivePPI', 'alpha': 0.003, 'beta': 0.3162277660168379, 'tau_rule': '0.5', 'development_deviance': 0.6769061590737862, 'converged': True, 'iterations': 5}}
Frozen final A-only models: A172__Lapatinib__replicate_1
Development selected: A172__Trametinib__replicate_1 {'CellOnly': {'method': 'CellOnly', 'alpha': 0.003, 'beta': 0.0, 'tau_rule': 'infinity', 'development_deviance': 0.6990046311207471, 'converged': True, 'iterations': 0}, 'FixedPPI': {'method': 'FixedPPI', 'alpha': 0.003, 'beta': 0.0017782794100389228, 'tau_rule': 'infinity', 'development_deviance'

In [6]:
import inspect
import hashlib
from pathlib import Path

if not getattr(rep, "_cell6_na_fix_v1", False):
    old_source = inspect.getsource(rep.evaluate_final)

    old_expression = (
        "meta.evaluation_eligible.to_numpy() & "
        "np.array([(a,d) in CONTEXTS for a,d in zip(meta.cell_line,meta.drug)])"
    )
    new_expression = (
        "meta.evaluation_eligible.to_numpy(dtype=bool, na_value=False) & "
        'pd.MultiIndex.from_frame(meta[["cell_line", "drug"]]).isin(CONTEXTS)'
    )

    if old_source.count(old_expression) != 1:
        raise RuntimeError(
            "The scoring function version is inconsistent with the patch; please preserve the scene, do not remove access locks or rerun parameter tuning."
        )

    patched_source = old_source.replace(old_expression, new_expression)

    audit = {
        "fix": "cell6_na_context_mask_v1",
        "original_function_sha256": hashlib.sha256(
            old_source.encode()
        ).hexdigest(),
        "patched_function_sha256": hashlib.sha256(
            patched_source.encode()
        ).hexdigest(),
        "scope": "NA-safe context mask only; no fitting, tuning or prediction changes",
    }
    rep.freeze(Path(EXPERIMENT_PATH) / "scoring_na_fix_v1.json", audit)

    exec(
        compile(patched_source, "<cell6_na_fix_v1>", "exec"),
        rep.__dict__,
    )
    rep._cell6_na_fix_v1 = True

print("Missing value filtering has been fixed, please continue running from Cell 6.")

Missing value filtering has been fixed, please continue running from Cell 6.


In [7]:
RESULTS=rep.evaluate_final(CONFIG,EXPERIMENT_PATH) if CONFIG["run_final_evaluation"] else None
if RESULTS is not None:
    cols=["cell_line","drug","replicate","method","n_cells","n_pairs","mean_poisson_deviance",
          "relative_deviance_percent","normalized_mse","alpha","beta","tau_rule"]
    print(RESULTS[cols].to_string(index=False))
else:
    print("Finally, B has not been scored; currently, only development/model preparation has been completed.")


Final B score block 1
Final B score block 2
Final B score block 3
Final B score block 4
Final B score block 5
Final B score block 6
Final B score block 7
Final B score block 8
Final B score block 9
Final B score block 10
Final B score block 11
Final B score block 12
Final B score block 13
Final B score block 14
Final B score block 15
Final B score block 16
Final B score block 17
Final B score block 18
Final B score block 19
Final B score block 20
Final B score block 21
Final B score block 22
Final B score block 23
Final B score block 24
Final B score block 25
Final B score block 26
Final B score block 27
Final B score block 28
Final B score block 29
Final B score block 30
Final B score block 31
Final B score block 32
Final B score block 33
Final B score block 34
Final B score block 35
Final B score block 36
Final B score block 37
Final B score block 38
Final B score block 39
Final B score block 40
Final B score block 41
Final B score block 42
Final B score block 43
Final B score block 

In [8]:
SUMMARY=rep.comparison_report(CONFIG,EXPERIMENT_PATH,RESULTS)
print(json.dumps(SUMMARY,ensure_ascii=False,indent=2))
print(__import__("pandas").read_csv(EXPERIMENT_PATH/"pipeline_difference_table.csv").to_string(index=False))


{
  "status": "four_context_baseline_scored",
  "experiment_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/step5/step5_four_contexts",
  "n_scope_cells": 228259,
  "n_scope_pairs": 570647500,
  "contexts": [
    [
      "A172",
      "Lapatinib"
    ],
    [
      "A172",
      "Trametinib"
    ],
    [
      "T98G",
      "Nintedanib"
    ],
    [
      "U87MG",
      "ZSTK474"
    ]
  ],
  "replicate_labels": [
    "replicate_1",
    "replicate_2"
  ],
  "cross_replicate_evaluation": "blocked_pending_provenance",
  "old_exact_numerical_reproduction": "not_verified_without_original_raw_outputs",
  "old_new_direct_error_comparison": "not_valid_different_protocols",
  "final_B_accessed": true,
  "requires_review_before_Task2": true,
  "new_all_contexts_claim": false
}
                item                                                                                               old_pipeline                                                  

In [9]:
if RESULTS is not None:
    plot_spec=importlib.util.spec_from_file_location("step5_plot",ROOT/"src/plot_reproduction.py")
    plot_module=importlib.util.module_from_spec(plot_spec)
    plot_spec.loader.exec_module(plot_module)
    FIGURE_PATH=ROOT/"figures"/(CONFIG["experiment_name"]+"_baseline.png")
    plot_module.plot_results(EXPERIMENT_PATH/"baseline_reproduction.csv",FIGURE_PATH)
    print("The chart has been generated from the result CSV：",FIGURE_PATH)


The chart has been generated from the result CSV： /home/featurize/work/biological_prior_validation/figures/step5_four_contexts_baseline.png


In [10]:
REVIEW_ZIP=ROOT/"results/summary"/(CONFIG["experiment_name"]+"_review_bundle.zip")
REVIEW_ZIP.parent.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(REVIEW_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EXPERIMENT_PATH.rglob("*")):
        if p.is_file() and p.suffix in [".json",".csv",".txt"]:
            z.write(p,p.relative_to(EXPERIMENT_PATH))
    report=ROOT/"reports"/(CONFIG["experiment_name"]+"_review.md")
    if report.exists():z.write(report,"reports/"+report.name)
    if RESULTS is not None and FIGURE_PATH.exists():z.write(FIGURE_PATH,"figures/"+FIGURE_PATH.name)
print("Please return this review compressed file:")
print(REVIEW_ZIP)


Please return this review compressed file:
/home/featurize/work/biological_prior_validation/results/summary/step5_four_contexts_review_bundle.zip
